## Install required libraries

In [1]:
pip install transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

## Load the Dataset, Model and Tokenizer

In [3]:
# 1. Load the dataset
dataset = load_dataset("knkarthick/dialogsum")

# 2. Load the tokenizer and model
model_name = "google/flan-t5-small"   # You can use 'flan-t5-base' or 'flan-t5-large' too
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Apply LoRA

In [4]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,                       # 'r' is the rank of the LoRA matrices. A lower rank means fewer additional parameters, balancing performance and efficiency.
    lora_alpha=32,             # Scaling factor to balance the update strength of the low-rank matrices.
    target_modules=["q", "v"], # Specifies which layers to apply LoRA to. "q" and "v" are the query and value projection layers in attention.
    lora_dropout=0.05,         # Dropout rate applied within the LoRA layers during training to help regularization.
    bias="none",               # Indicates whether to fine-tune bias parameters; "none" means we don't touch the bias terms.
    task_type=TaskType.SEQ_2_SEQ_LM  # Specifies the task type: in this case, it's sequence-to-sequence language modeling (e.g., summarization, translation).
)

# Apply LoRA configuration to the model
model = get_peft_model(model, lora_config)

## Preprocess the Dataset

In [ ]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = examples["dialogue"]
    targets = examples["summary"]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(targets, max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

## Prepare Data Collator 

In [6]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

## Define Training Arguments

In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./flan-t5-lora-dialogsum",        # Directory to save checkpoints and final model
    per_device_train_batch_size=8,                # Batch size per GPU core for training
    per_device_eval_batch_size=8,                 # Batch size per GPU core for evaluation
    logging_steps=100,                            # Log training metrics every 100 steps
    save_steps=500,                               # Save model checkpoint every 500 steps
    num_train_epochs=3,                           # Train the model for 3 full passes over the dataset
    save_total_limit=2,                           # Keep only the 2 most recent checkpoints to save disk space
    learning_rate=1e-4,                           # Learning rate for the optimizer
    bf16=True,                                    # Use bfloat16 precision if supported by hardware (more efficient than fp32)
    optim="adamw_torch",                          # Optimizer: AdamW implemented using PyTorch backend
    weight_decay=0.01,                            # Apply 0.01 L2 weight decay (regularization) to avoid overfitting
    push_to_hub=False,                            # Set to True if you want to push model to Hugging Face Hub
    report_to="tensorboard",                      # Log training metrics to TensorBoard for visualization
)

## Train the Model

In [8]:
from transformers import Trainer

trainer = Trainer(
    model=model,                                 # The LoRA-applied model to be trained
    args=training_args,                          # The training arguments defined earlier
    train_dataset=tokenized_datasets["train"],   # The training dataset (preprocessed and tokenized)
    eval_dataset=tokenized_datasets["validation"], # The validation dataset to monitor performance
    tokenizer=tokenizer,                         # The tokenizer used for preprocessing input and output
    data_collator=data_collator,                 # Handles dynamic padding and formatting for batches
)

trainer.train()  # Start training the model using the above settings

<ipython-input-8-8d8dbf269fb3>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
100,2.004600
200,1.669800
300,1.643000
400,1.580200
500,1.553500
600,1.558200
700,1.539200
800,1.544200
900,1.516400
1000,1.488200


TrainOutput(global_step=4674, training_loss=1.5041844437762004, metrics={'train_runtime': 1368.4583, 'train_samples_per_second': 27.315, 'train_steps_per_second': 3.416, 'total_flos': 5195756527706112.0, 'train_loss': 1.5041844437762004, 'epoch': 3.0})

## Save the Fine-Tuned Model

In [ ]:
model_path = "/content/Trained_Model/flan-t5-lora-dialogsum"  # Path to save the trained model
tokenizer_path = "/content/Trained_Model/flan-t5-lora-dialogsum"  # Path to save the tokenizer

# Save the trained model and tokenizer
model.save_pretrained(model_path)
tokenizer.save_pretrained(tokenizer_path)

('/content/Trained_Model/flan-t5-lora-dialogsum/tokenizer_config.json',
 '/content/Trained_Model/flan-t5-lora-dialogsum/special_tokens_map.json',
 '/content/Trained_Model/flan-t5-lora-dialogsum/spiece.model',
 '/content/Trained_Model/flan-t5-lora-dialogsum/added_tokens.json',
 '/content/Trained_Model/flan-t5-lora-dialogsum/tokenizer.json')

## Test Fine-Tuned Model using Training Dataset

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the saved model and tokenizer
model_path = "./flan-t5-lora-dialogsum"  # Replace with your own path if different
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

# Choose a sample from the validation set
sample = dataset["validation"][0]["dialogue"]
print("Dialogue:\n", sample)

# Tokenize and prepare input
inputs = tokenizer(sample, return_tensors="pt", truncation=True, max_length=512)

# Generate summary
summary_ids = model.generate(**inputs, max_length=128, num_beams=4)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\nGenerated Summary:\n", summary)